# 🔐 SSH Brute-Force Detector — Live Demo
**Student:** Jeeva J P  
**Roll No:** 727823TUCY018  
**Project:** SSH Brute-Force Detector  
**Date:** 2026-03-28

---
This notebook demonstrates all three distinct test cases of the SSH Brute-Force Detector tool.

In [ ]:
# student_name : Jeeva J P
# roll_number  : 727823TUCY018
# project_name : SSH Brute-Force Detector
# date         : 2026-03-28

import sys, os, datetime
sys.path.insert(0, '../code')
sys.path.insert(0, '../code/helper_modules')

from tool_main import SSHBruteForceDetector
from log_parser import generate_test_case_1, generate_test_case_2, generate_test_case_3, generate_test_case_4

ROLL_NUMBER = '727823TUCY018'
print(f'Roll No  : {ROLL_NUMBER}')
print(f'Timestamp: {datetime.datetime.now()}')
print('Libraries loaded ✓')

## Test Case 1 — Classic Brute-Force Attack
Single attacker IP (`192.168.1.100`) makes 12 rapid failed login attempts against `root` within 60 seconds.  
**Expected:** HIGH severity `BRUTE_FORCE_FAILED_PASSWORD` alert.

In [ ]:
# Generate TC1 log
tc1_path = '/tmp/tc1_demo.log'
generate_test_case_1(tc1_path)

# Show log contents
print('--- Log Contents (TC1) ---')
with open(tc1_path) as f:
    print(f.read())

In [ ]:
# Run detector
d1 = SSHBruteForceDetector(threshold=5, window_seconds=60)
d1.parse_log(tc1_path)
alerts1 = d1.detect()
d1.print_report()

print(f'Alerts found: {len(alerts1)}')
for a in alerts1:
    print(f"  → [{a['severity']}] {a['alert_type']} from {a['ip']} ({a['count']} attempts)")

## Test Case 2 — User Enumeration Scan
Attacker IP (`10.0.0.55`) probes 11 different usernames in rapid succession (`admin`, `root`, `test`, `oracle`, `postgres`, `ftp`, ...).  
**Expected:** HIGH severity `USER_ENUMERATION` alert.

In [ ]:
tc2_path = '/tmp/tc2_demo.log'
generate_test_case_2(tc2_path)

d2 = SSHBruteForceDetector(threshold=5, window_seconds=120)
d2.parse_log(tc2_path)
alerts2 = d2.detect()
d2.print_report()

print(f'Unique users tried: {alerts2[0]["unique_users"] if alerts2 else "N/A"}')

## Test Case 3 — Benign Mistype (True Negative)
User `alice` from `172.16.0.20` fails twice — but the attempts are 5 minutes apart.  
**Expected:** No alerts (failures do not exceed threshold within the sliding window).

In [ ]:
tc3_path = '/tmp/tc3_demo.log'
generate_test_case_3(tc3_path)

d3 = SSHBruteForceDetector(threshold=5, window_seconds=60)
d3.parse_log(tc3_path)
alerts3 = d3.detect()
d3.print_report()

assert len(alerts3) == 0, 'Expected 0 alerts for TC3'
print('✓ Correctly identified as benign — no false positive!')

## Test Case 4 — Distributed Attack
4 attacker IPs (`203.0.113.10-13`) each send 8 failed attempts, coordinating an attack with each IP staying at MEDIUM severity.  
**Expected:** 4 MEDIUM `BRUTE_FORCE_FAILED_PASSWORD` alerts, one per IP.

In [ ]:
tc4_path = '/tmp/tc4_demo.log'
generate_test_case_4(tc4_path)

d4 = SSHBruteForceDetector(threshold=5, window_seconds=60)
d4.parse_log(tc4_path)
alerts4 = d4.detect()
d4.print_report()

print(f'Total distributed alerts: {len(alerts4)}')
for a in alerts4:
    print(f"  → [{a['severity']}] {a['ip']} — {a['count']} attempts")

## Summary Table


In [ ]:
print(f'Roll No: 727823TUCY018  |  Timestamp: {datetime.datetime.now()}\n')
print(f'{"TC":<5} {"Description":<35} {"Alerts":<8} {"Result"}')
print('-' * 65)
rows = [
    ('TC1', 'Classic Brute-Force',       len(alerts1), 'HIGH alert raised ✓'),
    ('TC2', 'User Enumeration',          len(alerts2), 'HIGH alert raised ✓'),
    ('TC3', 'Benign Mistype (no alert)', len(alerts3), 'No false positive ✓'),
    ('TC4', 'Distributed Attack',        len(alerts4), '4 MEDIUM alerts ✓'),
]
for tc, desc, count, result in rows:
    print(f'{tc:<5} {desc:<35} {count:<8} {result}')